In [14]:
print("hello2222")

hello2222


In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()
# model = "claude-haiku-4-5"
# base_url = "http://127.0.0.1:8045"
# api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"

# base_url = "https://wzw.pp.ua"
# api_key="sk-RpLfnfZTIbvzsz8qVTcFmEHYdbt8m45HT544qmSpKJKNWITh"


# 模拟 Claude Code CLI (TypeScript SDK / Node.js) 的请求头
# claude_code_headers = {
#     "User-Agent": "claude-code/1.0.24",
#     "X-Stainless-Lang": "js",
#     "X-Stainless-Package-Version": "0.52.0",
#     "X-Stainless-OS": "MacOS",
#     "X-Stainless-Arch": "arm64",
#     "X-Stainless-Runtime": "node",
#     "X-Stainless-Runtime-Version": "v22.13.1",
#     "X-Stainless-Async": "async:promise",
# }

# client = Anthropic(api_key=api_key, base_url=base_url, default_headers=claude_code_headers)
# # model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"

base_url = "https://ai.hybgzs.com/claude"
api_key="sk-bTFxZYgcnhFm8ZG_zpRQg_W3HEfzFQB8voZGTJxWPqpi3dVl0x6W6bD4NTw"
client = Anthropic(api_key=api_key, base_url=base_url)
model = "claude-sonnet-4-6"


# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)



def chat(messages, system=None, temperature=1.0, stop_sequences=[], max_tokens=4000, tools=None):
    params = {
        "model": model,
        "max_tokens": max_tokens,  # 增加到 4000，支持更长的输出
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            if stop_sequences:
                for stop_seq in stop_sequences:
                    if stop_seq in block.text:
                        block.text = block.text.split(stop_seq)[0]
                        break   
    return message

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [16]:
# Tools function

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# get_current_datetime tool function
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    """
    在给定日期时间上加上指定时长，并返回格式化后的新日期时间字符串。
    使用 dateutil.relativedelta 自动处理所有边界情况（月末、闰年等）。
    
    参数:
    - datetime_str: 输入的日期时间字符串
    - duration: 要添加的时间量，可以是正数（用于未来日期）或负数（用于过去日期）
    - unit: 时间单位，可以是 'seconds'、'minutes'、'hours'、'days'、'weeks'、'months' 或 'years'
    - input_format: 输入日期时间的格式字符串，使用 Python 的 strptime 格式代码
    
    返回:
    - 格式化后的新日期时间字符串
    """
    date = datetime.strptime(datetime_str, input_format)

    # 使用 relativedelta 统一处理所有时间单位，自动处理边界情况
    unit_map = {
        "seconds": "seconds",
        "minutes": "minutes",
        "hours": "hours",
        "days": "days",
        "weeks": "weeks",
        "months": "months",
        "years": "years",
    }
    
    if unit not in unit_map:
        raise ValueError(f"Unsupported time unit: {unit}. Must be one of: {list(unit_map.keys())}")
    
    # relativedelta 会自动处理所有复杂的边界情况：
    # - 月份加减时的年份进位
    # - 月末日期（如 1月31日 + 1个月 → 2月28/29日）
    # - 闰年处理
    # - 年份加减时的日期保持
    delta_kwargs = {unit_map[unit]: duration}
    new_date = date + relativedelta(**delta_kwargs)

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\n 设置提醒 {timestamp}:\n{content}\n----")



pass

In [17]:
nowTime = get_current_datetime()
print("nowTime:",nowTime)
nextTime = add_duration_to_datetime(nowTime, 10, "days", "%Y-%m-%d %H:%M:%S")
set_reminder("吃药", nextTime)

nowTime: 2026-03-28 17:43:23
----
 设置提醒 Tuesday, April 07, 2026 05:43:23 PM:
吃药
----


In [18]:
# Tools schemas
get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": """根据指定格式返回当前日期和时间。
                此工具提供格式化的系统时间字符串，适用于时间戳记录、时间差计算或向用户显示当前时间。
                默认格式为 YYYY-MM-DD HH:MM:SS。""",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": """指定返回日期时间格式的字符串，使用 Python 的 strftime 格式代码。
                    例如：'%Y-%m-%d' 返回日期（YYYY-MM-DD），'%H:%M:%S' 返回时间（HH:MM:SS），'%B %d, %Y' 返回如 'May 07, 2025' 的日期。
                    默认为 '%Y-%m-%d %H:%M:%S'，返回完整时间戳如 '2025-05-07 14:32:15'。.""",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": """
              将指定的时间长度添加到日期时间字符串，并以详细格式返回结果日期时间。
              此工具将输入的日期时间字符串转换为 Python datetime 对象，
              在请求的时间单位中添加指定的时间长度，并返回格式化后的结果日期时间字符串。
              它处理各种时间单位，包括秒、分钟、小时、天、周、月和年，对月份和年份的计算进行特殊处理，以考虑不同月份长度和闰年。
              输出始终以详细格式返回，包括星期几、月份名称、日期、年份和带 AM/PM 指示符的时间（
              例如，'Thursday, April 03, 2025 10:30:00 AM'）。
            """,
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "要添加时间长度的输入日期时间字符串。应根据 input_format 参数进行格式化。",
            },
            "duration": {
                "type": "number",
                "description": "要添加到日期时间的时间量。可以是正数（用于未来日期）或负数（用于过去日期）。默认为 0。",
            },
            "unit": {
                "type": "string",
                "description": "时间长度的单位。必须是以下之一：'seconds'、'minutes'、'hours'、'days'、'weeks'、'months' 或 'years'。默认为 'days'。",
            },
            "input_format": {
                "type": "string",
                "description": "用于解析输入 datetime_str 的格式字符串，使用 Python 的 strptime 格式代码。例如，'%Y-%m-%d' 用于 ISO 格式日期，如 '2025-04-03'。默认为 '%Y-%m-%d'。",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": """创建一个定时提醒，将在指定时间使用提供的内容通知用户。
    此工具安排在提供的精确时间戳向用户发送通知。
    当用户希望在未来的某个时间点被提醒某事时，应使用此工具。
    提醒系统将存储内容和时间戳，然后在指定时间到达时通过用户首选的通知渠道（移动设备提醒、电子邮件等）触发通知。
    即使应用程序关闭或设备重启，提醒也会持久保存。
    用户可以依赖此功能来处理重要的时间敏感通知，如会议、任务、药物时间表或任何其他有时间限制的活动。
    """,
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "将在提醒通知中显示的消息文本。应包含用户希望被提醒的具体信息，例如'服药'、'加入团队视频通话'或'支付水电费'。",
            },
            "timestamp": {
                "type": "string",
                "description": """应触发提醒的确切日期和时间，格式为 ISO 8601 时间戳（YYYY-MM-DDTHH:MM:SS）或 Unix 时间戳。
                系统在内部处理所有时区处理，确保无论用户位于何处，提醒都会在正确的时间触发。
                用户可以简单地指定所需时间，而无需担心时区配置。""",
            },
        },
        "required": ["content", "timestamp"],
    },
}

In [19]:
import json


def run_tool(tool_name, tool_input):
    print(f"run_tool: {tool_name}, {tool_input}")
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [20]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema,
            ],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [21]:
messages = []
add_user_message(
    messages,
    "请为我的医生预约设置一个提醒，当前时间之后的第2000天",
)
run_conversation(messages)


run_tool: get_current_datetime, {}

run_tool: add_duration_to_datetime, {'datetime_str': '2026-03-28 17:43:27', 'duration': 2000, 'input_format': '%Y-%m-%d %H:%M:%S', 'unit': 'days'}

run_tool: set_reminder, {'content': '🏥 医生预约提醒：您有一个医生预约，请准时前往！', 'timestamp': '2031-09-18T17:43:27'}
----
 设置提醒 2031-09-18T17:43:27:
🏥 医生预约提醒：您有一个医生预约，请准时前往！
----
✅ **提醒设置成功！** 以下是详细信息：

| 项目 | 详情 |
|------|------|
| 📅 **当前时间** | 2026年3月28日 17:43:27 |
| ⏰ **提醒时间** | 2031年9月18日（星期四）下午 5:43:27 |
| 📝 **提醒内容** | 🏥 医生预约提醒：您有一个医生预约，请准时前往！ |
| 📆 **距今天数** | 2000 天 |

您的提醒已成功设定，系统将在 **2031年9月18日** 准时通知您医生预约的事宜。即使设备重启或应用关闭，提醒也会持久保存，请放心！


[{'role': 'user', 'content': '请为我的医生预约设置一个提醒，当前时间之后的第2000天'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='call_dac5a9da907a4cadabc54135', input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'call_dac5a9da907a4cadabc54135',
    'content': '"2026-03-28 17:43:27"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='call_3c81e49297ac4d849adf0fc0', input={'datetime_str': '2026-03-28 17:43:27', 'duration': 2000, 'input_format': '%Y-%m-%d %H:%M:%S', 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'call_3c81e49297ac4d849adf0fc0',
    'content': '"Thursday, September 18, 2031 05:43:27 PM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='call_51e17e64b9cf47b983bcd5d6', input={'content': '🏥 医生预约提醒：您有一个医生预约，请准时前往！', 'timestamp': '2031-09-18T17:43:27'}